<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/01_Data_Scientist/advanced/07_causal_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Causal Inference — Beyond Correlation

"Correlation is not causation" — but how do we measure causation from observational data? Causal inference gives us rigorous tools to answer "what would have happened if..."

**Topics:** Potential outcomes framework, Difference-in-Differences, Regression Discontinuity, Propensity Score Matching, DoWhy (causal graphs)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
import warnings; warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

print('Causal Inference Notebook')
print('Key packages: dowhy, econml, statsmodels')
print('Install: pip install dowhy econml statsmodels')

## 1. Potential Outcomes Framework (Rubin Causal Model)

In [ ]:
# Fundamental Problem of Causal Inference:
# We can only observe ONE potential outcome for each unit
# Y_i(1) if treated, Y_i(0) if control — NEVER both!

# Notation:
# T_i ∈ {0,1} — treatment indicator
# Y_i(0) — potential outcome under control
# Y_i(1) — potential outcome under treatment
# Y_i    — observed outcome = T_i*Y_i(1) + (1-T_i)*Y_i(0)
#
# Average Treatment Effect (ATE) = E[Y(1) - Y(0)]
# ATT = E[Y(1) - Y(0) | T=1]  (treated units)
# ATC = E[Y(1) - Y(0) | T=0]  (control units)

# Simulate a job training program
n = 1000

# Confounders: age and education affect BOTH treatment assignment AND outcome
age       = np.random.normal(35, 10, n).clip(18, 65)
education = np.random.choice([1, 2, 3, 4], n, p=[0.2, 0.4, 0.3, 0.1])  # 1=HS, 4=PhD

# True potential outcomes (never observed simultaneously)
# More educated people earn more regardless of training
Y0 = 30000 + 2000*education + 100*age + np.random.normal(0, 3000, n)
Y1 = Y0 + 5000 + 1000*education  # training effect: higher for educated

# TRUE ATE = 5000 + 1000 * E[education] ≈ 5000 + 1000*2.3 = ~7300
true_ate = (Y1 - Y0).mean()
print(f'True ATE (ground truth): ${true_ate:,.0f}')

# Treatment assignment: MORE educated and OLDER more likely to get training
# This creates SELECTION BIAS
prop_score = 1 / (1 + np.exp(-(0.3*education + 0.02*(age-35) - 1.5)))
treatment = np.random.binomial(1, prop_score, n)

# Observed outcome
Y_obs = treatment * Y1 + (1 - treatment) * Y0

df = pd.DataFrame({
    'treatment': treatment,
    'age': age,
    'education': education,
    'income': Y_obs,
    'Y0': Y0,  # in practice, we NEVER have this!
    'Y1': Y1,  # or this!
})

# Naive estimate: compare treated vs control directly
naive_ate = df[df.treatment==1].income.mean() - df[df.treatment==0].income.mean()

print(f'Naive ATE (biased):      ${naive_ate:,.0f}')
print(f'Bias:                    ${naive_ate - true_ate:,.0f}')
print(f'\nWhy biased? Selection bias:')
print(f'  Avg education (treated):  {df[df.treatment==1].education.mean():.2f}')
print(f'  Avg education (control):  {df[df.treatment==0].education.mean():.2f}')
print(f'  Already more educated → would earn more ANYWAY')

# Visualize selection bias
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, feature, label in zip(axes, ['education', 'age'], ['Education Level', 'Age']):
    ax.hist(df[df.treatment==0][feature], bins=30, alpha=0.6, density=True,
            color='steelblue', label='Control')
    ax.hist(df[df.treatment==1][feature], bins=30, alpha=0.6, density=True,
            color='orange', label='Treated')
    ax.set_xlabel(label); ax.set_ylabel('Density')
    ax.set_title(f'Covariate Distribution: {label}'); ax.legend()
plt.suptitle('Selection Bias: Treated ≠ Control on Confounders', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

## 2. Difference-in-Differences (DiD)

In [ ]:
# DiD: compare treated vs control groups BEFORE and AFTER treatment
# DiD estimate = (post_treated - pre_treated) - (post_control - pre_control)
# Key assumption: PARALLEL TRENDS — both groups would have evolved the same way

# Simulate a minimum wage policy
# State A raises minimum wage in 2020, State B does not
np.random.seed(42)
n_per_group = 500

# Pre-period (2019)
employment_pre_treated = np.random.normal(70, 5, n_per_group)
employment_pre_control = np.random.normal(70, 5, n_per_group)  # same baseline

# Common time trend (both groups)
common_trend = -2  # employment slightly falls for everyone

# Post-period (2020): treatment effect = +3 (wage increase helps employment)
true_did = 3
employment_post_treated = employment_pre_treated + common_trend + true_did + np.random.normal(0, 3, n_per_group)
employment_post_control = employment_pre_control + common_trend + np.random.normal(0, 3, n_per_group)

# Build panel data
panel = pd.DataFrame({
    'unit': np.tile(np.arange(n_per_group*2), 2),
    'treated': np.tile(np.concatenate([np.ones(n_per_group), np.zeros(n_per_group)]), 2),
    'post': np.repeat([0, 1], n_per_group*2),
    'employment': np.concatenate([
        employment_pre_treated, employment_pre_control,
        employment_post_treated, employment_post_control
    ])
})

# DiD estimate
means = panel.groupby(['treated', 'post'])['employment'].mean()
did_estimate = (means[1,1] - means[1,0]) - (means[0,1] - means[0,0])

print('=== Difference-in-Differences ===')
print(f'True treatment effect: {true_did}')
print()
print('Group means:')
print(f'  Treated, Pre:  {means[1,0]:.2f}')
print(f'  Treated, Post: {means[1,1]:.2f}')
print(f'  Control, Pre:  {means[0,0]:.2f}')
print(f'  Control, Post: {means[0,1]:.2f}')
print()
print(f'DiD = ({means[1,1]:.2f} - {means[1,0]:.2f}) - ({means[0,1]:.2f} - {means[0,0]:.2f})')
print(f'    = {means[1,1]-means[1,0]:.2f} - {means[0,1]-means[0,0]:.2f}')
print(f'    = {did_estimate:.2f}')

# Visualize
periods = ['Pre (2019)', 'Post (2020)']
plt.figure(figsize=(10, 5))
plt.plot(periods, [means[1,0], means[1,1]], 'o-', lw=2.5, ms=10, color='orange', label='Treated (State A)')
plt.plot(periods, [means[0,0], means[0,1]], 's--', lw=2.5, ms=10, color='steelblue', label='Control (State B)')

# Counterfactual (what treated would have done without treatment)
counterfactual_post = means[1,0] + (means[0,1] - means[0,0])  # parallel trend
plt.plot(periods, [means[1,0], counterfactual_post], '^:', lw=2, ms=10,
         color='orange', alpha=0.5, label='Counterfactual (parallel trend)')

plt.annotate('', xy=(1, means[1,1]), xytext=(1, counterfactual_post),
             arrowprops=dict(arrowstyle='<->', color='red', lw=2))
plt.text(1.02, (means[1,1] + counterfactual_post)/2, f'DiD={did_estimate:.1f}', color='red', fontsize=11)

plt.ylabel('Employment Rate'); plt.title('Difference-in-Differences\nMinimum Wage Policy')
plt.legend(); plt.tight_layout(); plt.show()

## 3. Propensity Score Matching

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Propensity score: P(T=1 | X) — probability of treatment given covariates
# Match each treated unit to a control unit with similar propensity score
# This balances covariates between groups

# Use the job training dataset from Section 1
features = ['age', 'education']
X_conf = df[features].values
T = df['treatment'].values
Y = df['income'].values

# Estimate propensity scores with logistic regression
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_conf)
lr = LogisticRegression(max_iter=1000)
lr.fit(X_scaled, T)
propensity_scores = lr.predict_proba(X_scaled)[:, 1]

df['propensity'] = propensity_scores

# 1-1 Nearest Neighbor Matching without replacement
from scipy.spatial.distance import cdist

treated_idx  = np.where(T == 1)[0]
control_idx  = np.where(T == 0)[0]

# Match on propensity score (with caliper = 0.1 * std of logit propensity)
logit_ps = np.log(propensity_scores / (1 - propensity_scores + 1e-9))
caliper = 0.1 * logit_ps.std()

matched_treated = []
matched_control = []
available_controls = list(control_idx)

for t_idx in treated_idx:
    if not available_controls:
        break
    distances = np.abs(logit_ps[t_idx] - logit_ps[available_controls])
    min_dist_idx = np.argmin(distances)
    
    if distances[min_dist_idx] <= caliper:
        matched_treated.append(t_idx)
        matched_control.append(available_controls[min_dist_idx])
        available_controls.pop(min_dist_idx)

n_matched = len(matched_treated)
print(f'Matched {n_matched} treated-control pairs (caliper={caliper:.3f})')

# Estimate ATT on matched sample
Y_treated_matched = Y[matched_treated]
Y_control_matched = Y[matched_control]
att_matched = (Y_treated_matched - Y_control_matched).mean()

# Standard error via bootstrap
n_boot = 1000
boot_atts = []
for _ in range(n_boot):
    idx = np.random.choice(n_matched, n_matched, replace=True)
    boot_atts.append((Y_treated_matched[idx] - Y_control_matched[idx]).mean())
att_se = np.std(boot_atts)

print(f'\nTreatment Effect Estimates:')
print(f'  True ATE:            ${true_ate:>8,.0f}')
print(f'  Naive (biased):      ${naive_ate:>8,.0f}')
print(f'  PSM ATT:             ${att_matched:>8,.0f}  (SE={att_se:.0f})')
print(f'  95% CI:              ${att_matched - 1.96*att_se:>8,.0f} to ${att_matched + 1.96*att_se:>8,.0f}')

# Check covariate balance after matching
def standardized_mean_diff(a, b):
    return (a.mean() - b.mean()) / np.sqrt((a.std()**2 + b.std()**2) / 2)

print(f'\nCovariate balance (Standardized Mean Difference):')
for feat in ['education', 'age']:
    smd_before = standardized_mean_diff(df[df.treatment==1][feat], df[df.treatment==0][feat])
    smd_after  = standardized_mean_diff(df.iloc[matched_treated][feat], df.iloc[matched_control][feat])
    print(f'  {feat:<12} before={smd_before:.3f}  after={smd_after:.3f}  {"✓ balanced" if abs(smd_after)<0.1 else "⚠ imbalanced"}')

## 4. Regression Discontinuity Design (RDD)

In [ ]:
# RDD: exploit arbitrary cutoffs in treatment assignment
# Example: scholarship eligibility at test score threshold
# Units just below vs just above cutoff are comparable

np.random.seed(42)
n_rdd = 2000
cutoff = 70  # test score cutoff for scholarship
bw = 10      # bandwidth around cutoff

# Running variable (test score)
test_score = np.random.uniform(40, 100, n_rdd)
treatment_rdd = (test_score >= cutoff).astype(int)

# True effect: scholarship improves graduation by 10pp
true_rdd_effect = 10

# Graduation probability: continuous function of score + jump at cutoff
# Y = 0.5 + 0.005*(score-70) + effect*T + noise
graduation = (0.4 + 0.006*(test_score - cutoff) +
              true_rdd_effect * treatment_rdd / 100 +
              np.random.normal(0, 0.05, n_rdd))
graduation = graduation.clip(0, 1)

df_rdd = pd.DataFrame({'score': test_score, 'treatment': treatment_rdd, 'graduation': graduation})

# RDD estimate: local linear regression on each side of cutoff
from numpy.polynomial import polynomial as P

# Focus on bandwidth around cutoff
rdd_sample = df_rdd[np.abs(df_rdd.score - cutoff) <= bw]

left  = rdd_sample[rdd_sample.treatment == 0]
right = rdd_sample[rdd_sample.treatment == 1]

def local_linear_at_cutoff(data, cutoff, side):
    """Fit local linear regression and evaluate at cutoff."""
    x = data.score.values - cutoff  # center at cutoff
    y = data.graduation.values
    coefs = np.polyfit(x, y, deg=1)
    return coefs[-1]  # value at x=0 (the cutoff)

y_left  = local_linear_at_cutoff(left, cutoff, 'left')
y_right = local_linear_at_cutoff(right, cutoff, 'right')
rdd_estimate = (y_right - y_left) * 100  # in percentage points

print('=== Regression Discontinuity Design ===')
print(f'Cutoff: test score = {cutoff}')
print(f'Bandwidth: ±{bw} points')
print(f'Sample in bandwidth: {len(rdd_sample)} units')
print(f'True RDD effect: {true_rdd_effect:.1f}pp')
print(f'Estimated effect: {rdd_estimate:.1f}pp')

# Visualize RDD
plt.figure(figsize=(12, 5))
# Scatter: raw data (binned means)
bins = np.arange(40, 101, 3)
df_rdd['bin'] = pd.cut(df_rdd.score, bins)
bin_means = df_rdd.groupby('bin')['graduation'].mean()
bin_centers = [(b.left + b.right)/2 for b in bin_means.index]
plt.scatter(bin_centers, bin_means, s=40, color='steelblue', zorder=5, label='Bin averages')

# Regression lines
x_left  = np.linspace(40, cutoff, 100)
x_right = np.linspace(cutoff, 100, 100)

coefs_left  = np.polyfit(left.score - cutoff, left.graduation, deg=1)
coefs_right = np.polyfit(right.score - cutoff, right.graduation, deg=1)

plt.plot(x_left,  np.polyval(coefs_left,  x_left  - cutoff), 'b-', lw=2.5)
plt.plot(x_right, np.polyval(coefs_right, x_right - cutoff), 'r-', lw=2.5)
plt.axvline(cutoff, color='black', ls='--', lw=2, label=f'Cutoff={cutoff}')

# Jump annotation
plt.annotate('', xy=(cutoff+0.5, y_right), xytext=(cutoff+0.5, y_left),
             arrowprops=dict(arrowstyle='<->', color='green', lw=2.5))
plt.text(cutoff+1, (y_right+y_left)/2, f'RDD={rdd_estimate:.1f}pp', color='green', fontsize=11)

plt.xlabel('Test Score'); plt.ylabel('Graduation Rate')
plt.title(f'Regression Discontinuity\nScholarship Eligibility at score={cutoff}')
plt.legend(); plt.tight_layout(); plt.show()

## 5. DoWhy — Causal Graphs & Structural Causal Models

In [ ]:
# DoWhy: Python library for causal inference with DAGs
# 4-step framework: Model → Identify → Estimate → Refute

print('=== DoWhy Causal Framework ===')
print('Install: pip install dowhy')

dowhy_code = '''
import dowhy
from dowhy import CausalModel

# Step 1: Model — specify causal graph as DAG
model = CausalModel(
    data=df,
    treatment='treatment',
    outcome='income',
    # Common causes (confounders)
    common_causes=['age', 'education'],
    # Or specify full DAG:
    # graph="""dag {
    #   education -> treatment; education -> income;
    #   age -> treatment; age -> income;
    #   treatment -> income; }"""
)
model.view_model()  # visualize the DAG

# Step 2: Identify — which adjustment sets make causal effect identifiable?
identified_estimand = model.identify_effect(proceed_when_unidentifiable=True)
print(identified_estimand)

# Step 3: Estimate — choose an estimator
# Propensity Score Matching
estimate_psm = model.estimate_effect(
    identified_estimand,
    method_name='backdoor.propensity_score_matching',
    target_units='ate'
)
print(f"PSM ATE: {estimate_psm.value:.2f}")

# Linear regression adjustment
estimate_lr = model.estimate_effect(
    identified_estimand,
    method_name='backdoor.linear_regression',
    target_units='ate'
)
print(f"Regression ATE: {estimate_lr.value:.2f}")

# Step 4: Refute — validate assumptions
# Random common cause: add random variable, should not change estimate much
refuter_rcc = model.refute_estimate(
    identified_estimand, estimate_psm,
    method_name='random_common_cause'
)
print(refuter_rcc)

# Placebo treatment: shuffle treatment, effect should go to ~0
refuter_placebo = model.refute_estimate(
    identified_estimand, estimate_psm,
    method_name='placebo_treatment_refuter',
    placebo_type='permute'
)
print(refuter_placebo)
'''
print(dowhy_code)

# Summarize the causal inference landscape
print()
print('=== Causal Inference Method Selector ===')
print()
methods = [
    ('RCT (A/B test)', 'Random assignment', 'Randomization', 'Gold standard when feasible'),
    ('DiD', 'Panel data, pre/post', 'Parallel trends', 'Policy evaluation, natural experiments'),
    ('RDD', 'Arbitrary cutoff exists', 'Continuity at cutoff', 'Scholarships, eligibility thresholds'),
    ('PSM', 'Observational, rich covariates', 'No unmeasured confounders', 'Healthcare, education, business'),
    ('IV', 'Valid instrument exists', 'Exclusion restriction', 'Economics, gene-environment studies'),
    ('DML', 'High-dimensional controls', 'CIA + smoothness', 'Modern econometrics, large datasets'),
]
print(f'{"Method":<15} {"When to use":<30} {"Key assumption":<30} {"Common applications"}')
print('-' * 110)
for method, when, assumption, apps in methods:
    print(f'{method:<15} {when:<30} {assumption:<30} {apps}')